# YOLO spatial prompt audit on Karpathy VALID

Attach MS COCO 2014, objectdetectionecache and csa-prompt-full-karpathy-v175. CPU only. Download and extract the review ZIP, then open index.html.

In [ ]:
from pathlib import Path
import numpy as np
def detection_index(source):
    if isinstance(source, dict) and 'data' in source:
        source = source['data']
    if not isinstance(source, dict):
        raise ValueError('Detector source must map filenames to records')
    result = {}
    for name, record in source.items():
        filename = Path(name).name
        if filename in result:
            raise ValueError('Duplicate detector basename: ' + filename)
        objects = record if isinstance(record, list) else record['objects']
        if not isinstance(objects, list):
            raise ValueError('Missing/malformed objects: ' + filename)
        for obj in objects:
            box = np.asarray(obj['bbox'], dtype=float)
            score = float(obj['conf'])
            if box.shape != (4,) or not np.isfinite(box).all() or (not np.isfinite(score)) or (not 0 <= score <= 1):
                raise ValueError('Nonfinite/invalid detector value: ' + filename)
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError('Nonpositive xyxy area: ' + filename)
        result[filename] = objects
    return result
def relation(a,b):
    x=min(a[2],b[2])-max(a[0],b[0])
    y=min(a[3],b[3])-max(a[1],b[1])
    h=min(a[3]-a[1],b[3]-b[1])
    w=min(a[2]-a[0],b[2]-b[0])
    if y>=.5*h and 0<=-x<=.35*h:
        return 'next to'
    if x>=.5*w and 0<=-y<=.5*h:
        return 'above' if a[3]<=b[1] else 'below'
    return None

from collections import Counter,defaultdict
from html import escape
from pathlib import Path
from random import Random
from PIL import Image,ImageDraw
import hashlib,json,shutil

ROOT=Path('/kaggle/input/datasets')
COCO=ROOT/'vuthetam/mscoco-2014'
OUT=Path('/kaggle/working/spatial_prompt_valid_review')
def checked(root,name,digest):
    paths=list(root.rglob(name))
    if len(paths)!=1:
        raise ValueError(f'Expected one {name} under {root}, found {len(paths)}')
    with paths[0].open('rb') as f:
        if hashlib.file_digest(f,'sha256').hexdigest()!=digest:
            raise ValueError(f'SHA256 mismatch: {paths[0]}')
    return paths[0]

coco=checked(COCO,'dataset_coco.json','2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d')
val={int(r['cocoid']):r['filename'] for r in json.loads(coco.read_bytes())['images'] if r['split']=='val'}
assert len(val)==len(set(val.values()))==5000
cache=checked(ROOT/'ducanh2403/objectdetectionecache','objectdetectioncache.json','498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c')
det=detection_index(json.loads(cache.read_bytes()))
links=checked(ROOT/'ducanh2403/csa-prompt-full-karpathy-v175','prompt_links.jsonl','e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0')
upt={}
with links.open(encoding='utf-8') as f:
    for line in f:
        r=json.loads(line)
        if r['group']=='val':
            cid=r['coco_id']
            assert cid not in upt and r['filename']==val[cid]
            upt[cid]={tuple(sorted((a['subject_roi'],a['object_roi']))) for a in r['relations']}
assert upt.keys()==val.keys()

names=set(val.values())
paths={}
for path in COCO.rglob('COCO_*.jpg'):
    if path.name in names:
        if path.name in paths:
            raise ValueError(f'Duplicate image {path.name}')
        paths[path.name]=path
if paths.keys()!=names:
    raise ValueError(f'Missing {len(names-paths.keys())} VALID JPEGs under {COCO}')

def noun(label):
    if label in ('skis','scissors'):
        return label
    return ('an' if label[0] in 'aeiou' else 'a')+' '+label

pools=defaultdict(list)
covered=set()
image_counts=Counter()
for cid,filename in sorted(val.items()):
    with Image.open(paths[filename]) as im:
        w,h=im.size
    boxes=[]
    ordered=sorted(enumerate(det.get(filename,[])),key=lambda pair:(-float(pair[1]['conf']),pair[0]))[:150]
    for roi,(_,obj) in enumerate(ordered):
        b=[min(limit,max(0.,float(v))) for v,limit in zip(obj['bbox'],(w,h,w,h),strict=True)]
        if float(obj['conf'])>=.5 and min(b[2]-b[0],b[3]-b[1])>=.03*min(w,h):
            label=' '.join(obj['label'].lower().replace('_',' ').split())
            boxes.append((roi,label,b))
    labels=Counter(label for _,label,_ in boxes)
    predicates=set()
    for si,sl,sb in boxes:
        if sl!='person':
            continue
        for oi,ol,ob in boxes:
            if ol=='person' or tuple(sorted((si,oi))) in upt[cid]:
                continue
            rel=relation(sb,ob)
            if rel:
                repeated=labels[sl]>1 or labels[ol]>1
                pools[rel,repeated].append(dict(coco_id=cid,filename=filename,predicate=rel,
                    roi=[si,oi],boxes=[sb,ob],repeated=repeated,has_upt_relation=bool(upt[cid]),
                    prompt=f'{noun(sl)} {rel} {noun(ol)}'))
                predicates.add(rel)
    image_counts.update(predicates)
    if predicates:
        covered.add(cid)

rng=Random(1411)
strata=[(rel,rep) for rel in ('next to','above','below') for rep in (True,False)]
counts={rel:len(pools[rel,True])+len(pools[rel,False]) for rel,_ in strata}
for key in strata:
    rng.shuffle(pools[key])
sample,seen=[],set()
for _ in range(4):
    for key in strata:
        pool=pools[key]
        while pool and pool[-1]['coco_id'] in seen:
            pool.pop()
        if pool:
            row=pool.pop()
            seen.add(row['coco_id'])
            sample.append(dict(row,id=len(sample)+1))

summary=dict(valid_images=len(val),missing_detector_rows=len(names-det.keys()),images_with_candidates=len(covered),
    images_without_upt_relation=sum(not r for r in upt.values()),
    images_without_upt_with_candidates=sum(not upt[cid] for cid in covered),
    candidates_by_predicate=counts,images_by_predicate=dict(image_counts),
    sample_by_predicate=dict(Counter(r['predicate'] for r in sample)),review_min=6,review_accuracy_min=.8)
(OUT/'images').mkdir(parents=True,exist_ok=False)
(OUT/'manifest.json').write_text(json.dumps(dict(split='val',summary=summary,sample=sample),indent=2),encoding='utf-8')
options='<option value="">Chua cham</option><option value="yes">Dung</option><option value="no">Sai</option><option value="unclear">Khong ro</option>'
cards=[]
for row in sample:
    i=row['id']
    with Image.open(paths[row['filename']]) as original:
        im=original.convert('RGB')
    w,h=im.size
    im.thumbnail((720,540))
    draw=ImageDraw.Draw(im)
    for b,color,tag in zip(row['boxes'],('orange','cyan'),('S','O'),strict=True):
        x,y,r,d=b
        rect=(x*im.width/w,y*im.height/h,min(r*im.width/w,im.width-1),min(d*im.height/h,im.height-1))
        draw.rectangle(rect,outline=color,width=3)
        draw.text((rect[0]+4,rect[1]+4),tag,fill=color,stroke_width=1,stroke_fill='black')
    im.save(OUT/'images'/f'{i}.jpg',quality=88)
    cards.append(f'<section data-id="{i}"><h2>#{i} COCO {row["coco_id"]}</h2><img src="images/{i}.jpg" alt="COCO S/O"><p><b>{escape(row["prompt"])}</b></p>'
                 f'<label>Hai box chi dung doi tuong? <select class="boxes">{options}</select></label> '
                 f'<label>Quan he giua S va O dung? <select class="relation">{options}</select></label></section>')
page='''<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ch&#7845;m prompt kh&#244;ng gian</title>
<style>body{max-width:900px;margin:30px auto;padding:0 16px;font:16px system-ui}section{border:1px solid #ccc;margin:20px 0;padding:16px}img{max-width:100%}</style>
<h1>Ch&#7845;m prompt kh&#244;ng gian</h1><p>S (cam) la chu the, O (xanh) la doi tuong. Above/below theo vi tri trong anh. Next to nghia la o canh nhau.</p>
<button id="save">Tai ket qua JSON</button>'''+''.join(cards)+'''
<script>
document.querySelector('#save').onclick=()=>{
 const records=[...document.querySelectorAll('section')].map(s=>({id:Number(s.dataset.id),boxes:s.querySelector('.boxes').value,relation:s.querySelector('.relation').value}));
 if(records.some(r=>!r.boxes||!r.relation)){alert('Con muc chua cham');return;}
 const a=document.createElement('a');
 a.href=URL.createObjectURL(new Blob([JSON.stringify({records},null,2)],{type:'application/json'}));
 a.download='spatial_prompt_labels.json';a.click();
};
</script></html>'''
(OUT/'index.html').write_text(page,encoding='utf-8')
print(json.dumps(summary,indent=2))
print('SEND:',shutil.make_archive(str(OUT),'zip',OUT))
